<a href="https://colab.research.google.com/github/raul94IA/LLAMA_META/blob/main/Copia_de_Hands_On_Prompt_Engineering_y_Sistemas_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **HANDS - ON: PROMPT ENGINEERING Y SISTEMAS RAG**

Una vez vista la masterclass ***Prompt Engineering y Sistemas RAG***, se proporciona el siguiente ***Colab*** para construir, en vivo, distintas estrategias de prompting y un mini sistema de RAG.

Usamos **Groq** para tener acceso a inferencia con GPU gratuita, sin necesidad de infraestructura propia.

De igual manera, se proporciona la **solución** de este notebook a través del siguiente [enlace](https://colab.research.google.com/drive/1EyWl33ZyAWuyMKXz_mr9aaNu3JF8e8vS?usp=sharing).

## **CONFIGURACIÓN DEL ENTORNO**

Para no exponer tu ***API key*** directamente en el código, Colab ofrece un panel de ***Secrets*** (ícono de llave en la barra lateral izquierda) donde puedes guardarla de forma segura, añadiendo un nombre asociado a la ***API key*** para guardarla dentro de una variable y usarla dentro del notebook.

In [1]:
# Instalar cliente de Groq y leer API key desde Colab Secrets
!pip install groq -q

import os
from groq import Groq
from google.colab import userdata

client = Groq(api_key=userdata.get('GROQ_API_KEY'))
print("Cliente de Groq inicializado correctamente.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 1.8 MB/s eta 0:00:00
Cliente de Groq inicializado correctamente.


## **ESTRATEGIAS DE PROMPTS**

### **ZERO-SHOT VS. FEW-SHOT**

Zero-shot es pedirle al modelo que haga algo sin darle ejemplos. Few-shot le muestra dos o tres ejemplos de entrada y salida antes de pedirle la tarea real. Vamos a comparar ambas estrategias con la misma tarea de clasificación.

In [2]:
# Prompt de clasificación en modo zero-shot
prompt_zero_shot = "Clasifica el sentimiento de esta reseña en Positivo, Negativo o Mixto: 'El envío llegó tarde pero el producto es excelente.' Respuesta muy breve y corta."

response_zero = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": prompt_zero_shot}]
)

print("Zero-shot:", response_zero.choices[0].message.content)

Zero-shot: Mixto


In [3]:
# Prompt de clasificación en modo few-shot
#Nota personal: Las triples comillas indican cadena de documentacion que detecta saltos y espacios
prompt_few_shot = """Clasifica el sentimiento de cada reseña como Positivo, Negativo o Mixto.

Reseña: "Me encantó, llegó rápido y en perfecto estado."
Sentimiento: Positivo

Reseña: "Nunca llegó mi pedido, pésimo servicio."
Sentimiento: Negativo

Reseña: "El envío llegó tarde pero el producto es excelente."
Sentimiento:"""

response_few = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": prompt_few_shot}]
)

print("Few-shot:", response_few.choices[0].message.content)
# El formato few-shot suele acotar mejor la salida a una sola palabra de la categoría esperada

Few-shot: Sentimiento: Mixto


### **CHAIN-OF-THOUGHT**

Chain-of-thought le pide al modelo mostrar su razonamiento paso a paso antes de la respuesta final, algo que mejora notablemente el desempeño en problemas de lógica.

In [4]:
# Razonamiento paso a paso (chain-of-thought)
problema = (
    "Un tren sale de la ciudad A a 80 km/h. Dos horas después, otro tren sale de la misma "
    "ciudad hacia el mismo destino a 120 km/h. ¿Cuánto tiempo tarda el segundo tren en "
    "alcanzar al primero? Muestra tu razonamiento paso a paso antes de dar la respuesta final."
)

response_cot = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": problema}]
)

print(response_cot.choices[0].message.content)

**Razón paso a paso**

1. **Datos iniciales**  
   - Velocidad del primer tren \(v_1 = 80 \text{ km/h}\).  
   - Velocidad del segundo tren \(v_2 = 120 \text{ km/h}\).  
   - El segundo tren parte 2 h después del primero.

2. **Distancia inicial de ventaja**  
   El primer tren recorre en esos 2 h:  
   \[
   d_{\text{ventaja}} = v_1 \times 2 \text{ h} = 80 \times 2 = 160 \text{ km}
   \]

3. **Ecuación de las distancias cuando el segundo tren llega al primero**  
   Sea \(t\) el tiempo (en horas) que el segundo tren recorre después de su salida.

   - Distancia que recorre el segundo tren en ese tiempo:  
     \[
     d_2 = v_2 \, t = 120 t
     \]

   - El primer tren, que ya había estado viajando 2 h más, habrá recorrido:  
     \[
     d_1 = v_1 \,(t+2) = 80(t+2) = 80t + 160
     \]

4. **Condición de encuentro**  
   Cuando el segundo tren alcanza al primero, sus distancias desde A son iguales:
   \[
   d_2 = d_1 \quad\Longrightarrow\quad 120t = 80t + 160
   \]

5. **Resolver para

### **EL LÍMITE DEL PROMPT: LO QUE EL MODELO NUNCA VIO**

Ninguna técnica de prompting le da información nueva al modelo. Si le preguntamos algo que no pudo haber visto en su entrenamiento, puede alucinar una respuesta que suene convincente pero no sea real.

In [5]:
# Preguntar algo que el modelo no pudo haber visto en su entrenamiento y observar si alucina
prompt_desconocido = (
    "¿Cuál fue el resultado de la final del hackathon interno de DEV.F del 14 de agosto de "
    "2026? Respuesta muy breve y corta."
)

response_alucinacion = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": prompt_desconocido}]
)

print(response_alucinacion.choices[0].message.content)

Lo siento, pero no puedo responder esa pregunta.


## **RAG: BUSCAR ANTES DE RESPONDER**

RAG separa el proceso en dos pasos: primero un sistema de búsqueda encuentra los fragmentos más relevantes de una base de conocimiento propia (usando *embeddings* y similitud semántica); después, esos fragmentos se le entregan al modelo junto con la pregunta para generar la respuesta.

In [6]:
# Instalar sentence-transformers
!pip install sentence-transformers --quiet

from sentence_transformers import SentenceTransformer
import numpy as np

In [7]:
# Definir la base de conocimiento (política de devoluciones) y generar sus embeddings
modelo_embeddings = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

documentos = [
    "Las devoluciones se aceptan hasta 30 días después de la compra, con el producto en su "
    "empaque original.",
    "Los envíos internacionales no tienen devolución gratuita; el cliente cubre el costo de "
    "envío de regreso.",
    "Los productos en oferta o liquidación no son elegibles para devolución, solo para "
    "cambio de talla."
]

embeddings_documentos = modelo_embeddings.encode(documentos)
print("Embeddings generados:", embeddings_documentos.shape)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

... (4 outputs left out)

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embeddings generados: (3, 384)


In [8]:
# Definir una función que calcule la similitud entre la pregunta y cada fragmento, y regrese el más relevante
# Entiendo que se crea la funcion buscar fragmento, a la cual le damos una pregunta y nos devuelve el fragmento de los documentos mas alineado a la pregunta.
#Entiendo que embeddings_pregunta obtiene los vectores de la pregunta, similitudes obtiene la medida de similitud entre los embeddings de la pregunta y los embeddings de cada
#fragmento del documento. indice_mas_similar determina cual de los fragmentos es el mas alineado a la pregunta. La funcion regresa el fragmento del documento con la similitud mas alta.
def buscar_fragmento(pregunta):
    embedding_pregunta = modelo_embeddings.encode([pregunta])
    similitudes = np.dot(embeddings_documentos, embedding_pregunta.T).flatten()
    indice_mas_similar = np.argmax(similitudes)
    return documentos[indice_mas_similar]

pregunta = "¿Puedo devolver algo que compré en oferta?"
fragmento = buscar_fragmento(pregunta)
print("Fragmento recuperado:", fragmento)

Fragmento recuperado: Los productos en oferta o liquidación no son elegibles para devolución, solo para cambio de talla.


In [9]:
# Enviar la pregunta junto con el fragmento recuperado al modelo y mostrar la respuesta con RAG
prompt_rag = f"""Responde la pregunta del cliente usando SOLO la siguiente política de la tienda. Si la política no cubre la pregunta, dilo claramente.

Política: {fragmento}

Pregunta: {pregunta}

Respuesta muy breve y corta."""

response_rag = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": prompt_rag}]
)

print(response_rag.choices[0].message.content)

No, los artículos en oferta no se pueden devolver, solo se pueden cambiar de talla.


# **CHALLENGE: ASISTENTE DE POLÍTICAS CON RAG**

Una vez visto el ***Hands-On: Prompt Engineering y Sistemas RAG***, se presenta el siguiente reto para que el alumnado pueda repasar y reforzar lo aprendido dentro de la clase.

Se construirá un pequeño asistente con RAG sobre un documento propio, comparando la respuesta **con RAG** contra la respuesta **sin RAG** para la misma pregunta. En esta solución se usa como base de conocimiento el propio reglamento de evaluación del curso IA Aplicada con Llama.

**IMPORTANTE:** Para su revisión, **es indispensable que los apartados anteriores se encuentren llenados con el código visto durante la sesión.**

## **INSTRUCCIONES:**

**1. Define tu base de conocimiento y genera embeddings:**

* Lee la API key previamente configurada desde **Colab Secrets** e instala/importa las librerías necesarias.

* Construye una lista llamada `documentos` con 3 fragmentos de un documento real de tu propio contexto (reglamento, políticas, FAQs, etc.) y genera sus embeddings con `sentence-transformers`.

In [10]:
# Leer API key, instalar e importar librerías
!pip install groq -q

import os
from groq import Groq
from google.colab import userdata

client = Groq(api_key=userdata.get('GROQ_API_KEY'))
print("Cliente de Groq inicializado correctamente.")

!pip install sentence-transformers --quiet

from sentence_transformers import SentenceTransformer
import numpy as np

Cliente de Groq inicializado correctamente.


In [13]:
# Definir la lista documentos y generar sus embeddings
modelo_embeddings = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

documentos = [
    "El curso se divide en dos modulos: IA aplicada con modelos abiertos "
    "y Automatizacion Inteligente con WhatsApp.",
    "Cada modulo consta de 4 lecturas, 3 retos practicos y un proyecto "
    "llamado Hackathon.",
    "El alumno debe entragar por lo menos uno de los dos proyectos Hackathon para aprobar el curso."
]

embeddings_documentos = modelo_embeddings.encode(documentos)
print("Embeddings generados:", embeddings_documentos.shape)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Embeddings generados: (3, 384)


**2. Recupera el fragmento relevante:** Define una función `buscar_fragmento(pregunta)` que calcule la similitud coseno y regrese el fragmento más relevante para una pregunta dada.

In [14]:
# Definir la función buscar_fragmento
def buscar_fragmento(pregunta):
    embedding_pregunta = modelo_embeddings.encode([pregunta])
    similitudes = np.dot(embeddings_documentos, embedding_pregunta.T).flatten()
    indice_mas_similar = np.argmax(similitudes)
    return documentos[indice_mas_similar]

pregunta = "¿Puedo aprobar el curso si me falto entregar uno de los dos proyectos Hackathon?"
fragmento = buscar_fragmento(pregunta)
print("Fragmento recuperado:", fragmento)

Fragmento recuperado: El alumno debe entragar por lo menos uno de los dos proyectos Hackathon para aprobar el curso.


**3. Genera la respuesta sin RAG:** Envía una pregunta real sobre tu documento directamente al modelo (sin ningún fragmento de contexto) y guarda la respuesta en `respuesta_sin_rag`.

In [16]:
# Consultar la pregunta sin RAG y guardar el resultado en respuesta_sin_rag

respuesta_sin_rag_1 = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": pregunta}]
)
respuesta_sin_rag = respuesta_sin_rag_1.choices[0].message.content

print(respuesta_sin_rag)

Claro, es comprensible que te sientas preocupado. La respuesta depende principalmente de los requisitos específicos de tu curso y de cómo se haya estructurado la evaluación. A continuación tienes una guía práctica que te ayudará a decidir qué pasos seguir:

---

## 1. **Revisa el plan de estudios y la política de evaluaciones**

- **Syllabus**: Busca la sección donde se detalla la distribución de puntos.  
  - Si cada proyecto Hackathon vale, por ejemplo, 25 % de la nota final, faltar uno puede significar perder la mitad de esa categoría.  
  - Si la entrega del proyecto es obligatoria para obtener créditos (es decir, sin él no hay calificación), es probable que no puedas aprobar sin completarlo.

- **Política de entregas tardías o excusas**: Muchos cursos permiten entregas tardías con penalizaciones o aceptan excusas justificadas (enfermedad, problemas personales, etc.).  

---

## 2. **Evalúa tus otras calificaciones**

- **¿Qué tan cerca estás de la nota mínima de aprobación?**  
  

**4. Genera la respuesta con RAG:** Recupera el fragmento relevante con tu función y envía la pregunta junto con ese fragmento al modelo. Guarda la respuesta en `respuesta_con_rag`.

In [17]:
# Consultar la pregunta con RAG y guardar el resultado en respuesta_con_rag
pregunta_con_rag = f"""Responde la pregunta del cliente usando SOLO la siguiente política de la tienda. Si la política no cubre la pregunta, dilo claramente.

Política: {fragmento}

Pregunta: {pregunta}

Respuesta muy breve y corta."""

response_rag = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": pregunta_con_rag}]
)

respuesta_con_rag = response_rag.choices[0].message.content

print(respuesta_con_rag)

Sí, basta con entregar al menos uno de los dos proyectos Hackathon.


**5. Compara y concluye:** Imprime ambas respuestas y concluye cuál de las dos evitó mejor una alucinación o dio una respuesta más precisa.

In [19]:
# Mostrar ambas respuestas para comparar
# Sin RAG el modelo no puede dar una respuesta, ya que no sabe ni siquiera a que curso nos referimos y aunque supiera, dificilmente tenga registro del curso en sus bases de datos
#o entrenamientos, por lo que el momento se limita a dar instrucciones al alumno para que obtenga infomacion sobre su curso desde otras fuentes.
print("Respuesta con RAG:", respuesta_con_rag)
print("Respuesta sin RAG:", respuesta_sin_rag)

Respuesta con RAG: Sí, basta con entregar al menos uno de los dos proyectos Hackathon.
Respuesta sin RAG: Claro, es comprensible que te sientas preocupado. La respuesta depende principalmente de los requisitos específicos de tu curso y de cómo se haya estructurado la evaluación. A continuación tienes una guía práctica que te ayudará a decidir qué pasos seguir:

---

## 1. **Revisa el plan de estudios y la política de evaluaciones**

- **Syllabus**: Busca la sección donde se detalla la distribución de puntos.  
  - Si cada proyecto Hackathon vale, por ejemplo, 25 % de la nota final, faltar uno puede significar perder la mitad de esa categoría.  
  - Si la entrega del proyecto es obligatoria para obtener créditos (es decir, sin él no hay calificación), es probable que no puedas aprobar sin completarlo.

- **Política de entregas tardías o excusas**: Muchos cursos permiten entregas tardías con penalizaciones o aceptan excusas justificadas (enfermedad, problemas personales, etc.).  

---

#